In [11]:
import pandas as pd
import re
import os

# 1. Load your raw dataset from the established path
try:
# Notice that 'github.com' becomes 'raw.githubusercontent.com' and we remove '/blob/'
  df = pd.read_csv('https://raw.githubusercontent.com/sean-seah/HPDP/main/2526/project/p2/submission/DataAnalysis/data/raw_data/Malaysian_Telecommunication_Google%20Play_reviews.csv')
  print(f"Successfully loaded dataset! Total rows: {len(df)}")
except FileNotFoundError:
    print("Error: Check your file path! Ensure the CSV is in 'data/raw_data/'")

# 2. Convert Star Ratings into Categorical Sentiment Labels
def map_score_to_sentiment(score):
    if score <= 2:
        return 'Negative'
    elif score == 3:
        return 'Neutral'
    else:
        return 'Positive'

df['sentiment'] = df['score'].apply(map_score_to_sentiment)

# 3. Create a Malaysian-Context NLP Text Cleaning Function
def clean_malaysian_text(text):
    if not isinstance(text, str):
        return ""

    # Lowercase everything to standardize words
    text = text.lower()

    # Remove web links and URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove mentions (@user) and hashtags (#topic)
    text = re.sub(r'@\w+|#\w+', '', text)

    # Keep letter-characters, numbers, and spaces (strips punctuation & excessive emojis)
    # Note: We keep numbers because terms like "4g", "5g", "rm45" carry heavy technical context in telco reviews!
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    # Collapse multiple consecutive spaces down to a single space
    text = " ".join(text.split())

    return text

# Apply cleaning matrix to the review content
df['cleaned_content'] = df['content'].apply(clean_malaysian_text)

# 4. Filter out any rows where the text became empty after cleaning
df = df[df['cleaned_content'].str.strip() != ""]

# 5. Export this dataset for your model training phase
os.makedirs('C:/Users/jingy/Downloads/data/', exist_ok=True)

# Save directly into 'data'
df.to_csv('C:/Users/jingy/Downloads/data/cleaned_data.csv', index=False)
print("Data preprocessing complete! Exported to 'data/cleaned_data.csv'")

# Quick sanity check on local data formatting
print("\n--- Preprocessing Preview ---")
print(df[['provider', 'score', 'sentiment', 'cleaned_content']].head(3))

Successfully loaded dataset! Total rows: 159444
Data preprocessing complete! Exported to 'data/cleaned_data.csv'

--- Preprocessing Preview ---
  provider  score sentiment                                    cleaned_content
0    maxis      1  Negative  terrible app when i want to pay the bill using...
1    maxis      1  Negative  sooo unstable buggy and slow not surprising fr...
2    maxis      1  Negative  the app took so very long to sync its better t...
